# Tutorial: Ensemble Methods — The House Price Race

> **Type:** Tutorial. Follow along step by step.  
> **Time:** ~2.5 hours  
> **Prerequisites:** Module 11 (Supervised Learning Overview)  
> **What you'll build:** A benchmark of four ensemble methods on a synthetic house price dataset, with feature importance plots, an ensemble-size tuning curve, and a simple stacking ensemble.

---

## What We're Building

We'll generate a realistic synthetic house price dataset with 15 features (a mix of numeric and categorical). Then we'll race four ensemble methods against each other: Random Forest, Gradient Boosting, XGBoost, and LightGBM. You'll see how they differ in speed, accuracy, and what they say about feature importance. We'll finish by building a simple stacking ensemble, and checking whether it beats the best single model. (Spoiler: here it doesn't, and that's a lesson in itself.)

## Step 0: Setup

In [ ]:
# Install if needed
# !pip install xgboost lightgbm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Setup complete.')

## Step 1: Generate the Dataset

We'll build a synthetic house price dataset with realistic feature relationships.

In [ ]:
def make_house_prices(n=2000, seed=42):
    rng = np.random.default_rng(seed)
    
    n_rooms    = rng.integers(2, 9, n)                        # 2–8 rooms
    sqft       = rng.integers(600, 4000, n)                   # square footage
    lot_size   = rng.integers(1000, 20000, n)                 # sq ft lot
    age        = rng.integers(0, 80, n)                       # years old
    bathrooms  = np.clip(n_rooms // 2 + rng.integers(-1, 2, n), 1, 5)
    garage     = rng.choice([0, 1, 2], n, p=[0.2, 0.5, 0.3]) # garage spaces
    pool       = rng.choice([0, 1], n, p=[0.8, 0.2])         # has pool?
    floors     = rng.integers(1, 4, n)                        # number of floors
    school_dist = rng.uniform(0.1, 10, n)                     # miles to school
    crime_rate  = rng.uniform(0, 10, n)                       # crime index
    
    neighborhood = rng.choice(['downtown', 'suburbs', 'rural', 'waterfront'], n,
                               p=[0.25, 0.45, 0.15, 0.15])
    condition    = rng.choice(['poor', 'fair', 'good', 'excellent'], n,
                               p=[0.1, 0.25, 0.45, 0.2])
    style        = rng.choice(['ranch', 'colonial', 'contemporary', 'craftsman'], n)
    heating      = rng.choice(['gas', 'electric', 'oil'], n, p=[0.6, 0.3, 0.1])
    
    # Price = non-linear combination of features with noise
    nbhd_premium = {'downtown': 1.1, 'suburbs': 1.0, 'rural': 0.7, 'waterfront': 1.5}
    cond_factor  = {'poor': 0.75, 'fair': 0.9, 'good': 1.0, 'excellent': 1.15}
    
    price = (
        80 * sqft
        + 12000 * n_rooms
        + 8000 * bathrooms
        + 20000 * garage
        + 30000 * pool
        - 500 * age
        + 2 * lot_size
        - 8000 * school_dist
        - 5000 * crime_rate
        + rng.normal(0, 20000, n)  # noise
    )
    price *= np.array([nbhd_premium[nb] for nb in neighborhood])
    price *= np.array([cond_factor[c] for c in condition])
    price = np.maximum(price, 50000)  # no negative prices
    
    return pd.DataFrame({
        'sqft': sqft, 'n_rooms': n_rooms, 'bathrooms': bathrooms,
        'lot_size': lot_size, 'age': age, 'garage': garage,
        'pool': pool, 'floors': floors, 'school_dist': school_dist,
        'crime_rate': crime_rate, 'neighborhood': neighborhood,
        'condition': condition, 'style': style, 'heating': heating,
        'price': price.astype(int)
    })

df = make_house_prices(n=2000)
print(f'Dataset shape: {df.shape}')
print(f'Price range: ${df["price"].min():,} – ${df["price"].max():,}')
df.head()

## Step 2: Prepare Features

These tree ensembles (RF, XGBoost, LightGBM) need numeric inputs here, so we'll label-encode the categorical columns.

In [ ]:
feature_cols = [c for c in df.columns if c != 'price']
cat_cols = ['neighborhood', 'condition', 'style', 'heating']

df_enc = df.copy()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    encoders[col] = le

X = df_enc[feature_cols].values
y = df_enc['price'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Features: {feature_cols}')

## Step 3: Train and Benchmark Four Ensemble Methods

In [ ]:
models = {
    'Random Forest':       RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=RANDOM_SEED),
    'XGBoost':             xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=RANDOM_SEED, verbosity=0),
    'LightGBM':            lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=RANDOM_SEED, verbose=-1),
}

results = []
fitted_models = {}

for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0
    
    y_pred = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2, 'Train Time (s)': train_time})
    fitted_models[name] = model
    print(f'{name:<22}  RMSE=${rmse:>8,.0f}  R²={r2:.4f}  Time={train_time:.2f}s')

results_df = pd.DataFrame(results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colours = ['#4E79A7', '#F28E2B', '#E15759', '#59A14F']

# RMSE
axes[0].bar(results_df['Model'], results_df['RMSE'] / 1000, color=colours, alpha=0.85)
axes[0].set_title('RMSE (lower is better)', fontweight='bold')
axes[0].set_ylabel('RMSE ($000s)')
axes[0].tick_params(axis='x', rotation=20)

# R²
axes[1].bar(results_df['Model'], results_df['R²'], color=colours, alpha=0.85)
axes[1].set_title('R² Score (higher is better)', fontweight='bold')
axes[1].set_ylim(0.9, 1.0)
axes[1].tick_params(axis='x', rotation=20)

# Training time
axes[2].bar(results_df['Model'], results_df['Train Time (s)'], color=colours, alpha=0.85)
axes[2].set_title('Training Time (seconds)', fontweight='bold')
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Ensemble Method Benchmark — House Price Prediction', fontweight='bold')
plt.tight_layout()
plt.savefig('ensemble_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart description.** Three bar charts comparing four ensembles on house prices. RMSE in thousands of dollars: random forest about 45.5, gradient boosting 28.6, XGBoost 28.9 and LightGBM 28.3. R², with the axis starting at 0.9: gradient boosting 0.939, XGBoost 0.938 and LightGBM 0.940. Random forest's bar doesn't show because its R² is 0.845, below the axis. Training time: gradient boosting is slowest at about 0.5 seconds. LightGBM is fastest at about 0.05.

> **What just happened?** LightGBM was both the fastest (0.06s) and the most accurate (RMSE ~$28,300). The three boosting methods finished close together on accuracy, well ahead of Random Forest. Speed varied more than you might expect: XGBoost was the slowest here, and Random Forest, which trains its trees in parallel with `n_jobs=-1`, was faster than both sklearn's Gradient Boosting and XGBoost. Exact timings depend on your machine and library versions, so treat the speed ranking as one measurement, not a law.

## Step 4: Feature Importance Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, (name, model) in zip(axes, fitted_models.items()):
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    top_n = 10
    top_idx = indices[:top_n]
    top_features = [feature_cols[i] for i in top_idx]
    top_importance = importances[top_idx]
    
    ax.barh(top_features[::-1], top_importance[::-1], color='#4E79A7', alpha=0.85)
    ax.set_title(f'{name} — Feature Importance', fontweight='bold')
    ax.set_xlabel('Importance (impurity-based)')

plt.suptitle('Feature Importance by Model\n(all models agree: sqft and n_rooms dominate)', 
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Print Random Forest's importances as text (the book quotes these directly)
rf_importances = fitted_models['Random Forest'].feature_importances_
rf_order = np.argsort(rf_importances)[::-1]
print('Random Forest feature importances:')
for i in rf_order:
    print(f'  {feature_cols[i]:<15} {rf_importances[i]:.3f}')

**Chart description.** Four horizontal bar charts of the top 10 feature importances, one per model. All four rank sqft first. Random forest, gradient boosting and XGBoost put neighborhood second. LightGBM's importances are split counts, not impurity, and its ranking differs: crime_rate, school_dist and lot_size come next. The models agree on the top feature and disagree after that.

> **What just happened?** The models agree that `sqft` dominates, because we designed the price formula that way. After that the rankings differ a little: Random Forest, for example, puts `neighborhood` second, ahead of `n_rooms`. Feature importance is how you explain a model's decisions to a non-technical stakeholder, but different importance methods can rank the minor features differently.

## Step 5: How Does Ensemble Size Affect Accuracy?

In [ ]:
n_estimators_range = [5, 10, 25, 50, 100, 200, 300, 500]
rf_rmse = []
lgb_rmse = []

for n in n_estimators_range:
    # Random Forest
    rf = RandomForestRegressor(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_rmse.append(root_mean_squared_error(y_test, rf.predict(X_test)))
    
    # LightGBM
    lg = lgb.LGBMRegressor(n_estimators=n, learning_rate=0.1, max_depth=4, 
                            random_state=RANDOM_SEED, verbose=-1)
    lg.fit(X_train, y_train)
    lgb_rmse.append(root_mean_squared_error(y_test, lg.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, np.array(rf_rmse) / 1000, 'o-', label='Random Forest', color='#4E79A7', linewidth=2)
plt.plot(n_estimators_range, np.array(lgb_rmse) / 1000, 's-', label='LightGBM', color='#59A14F', linewidth=2)
plt.xlabel('n_estimators (number of trees)')
plt.ylabel('RMSE ($000s)')
plt.title('Accuracy vs. Ensemble Size', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('ensemble_size_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key insight: accuracy improves quickly, then plateaus. Beyond ~100-200 trees you\'re paying compute cost for diminishing returns.')

print('RF RMSE by n_estimators:', {n: round(r) for n, r in zip(n_estimators_range, rf_rmse)})
print('LightGBM RMSE by n_estimators:', {n: round(r) for n, r in zip(n_estimators_range, lgb_rmse)})

**Chart description.** RMSE in thousands of dollars against the number of trees, from 5 to 500, for random forest and LightGBM. The random forest starts at about 50 and flattens at about 45.5 by 200 trees. LightGBM starts much worse at about 88 with 5 trees, drops fast, passes the forest at about 40 trees and flattens at about 28 by 200 trees. More trees beyond 200 change little for either.

## Step 6: Build a Stacking Ensemble

In [ ]:
# Stacking: two base models + a Ridge meta-learner
# cross_val_predict gives out-of-fold predictions (prevents leakage)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

base_models = {
    'rf':  RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    'lgb': lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=RANDOM_SEED, verbose=-1),
}

# Step 1: Get out-of-fold predictions for training the meta-learner
oof_preds = {}
test_preds = {}

for name, model in base_models.items():
    oof_preds[name] = cross_val_predict(model, X_train, y_train, cv=kf)
    model.fit(X_train, y_train)  # fit on full train for test predictions
    test_preds[name] = model.predict(X_test)

# Step 2: Build meta-features
X_meta_train = np.column_stack(list(oof_preds.values()))
X_meta_test  = np.column_stack(list(test_preds.values()))

# Step 3: Train meta-learner
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta_train, y_train)
y_pred_stack = meta_model.predict(X_meta_test)

stack_rmse = root_mean_squared_error(y_test, y_pred_stack)
stack_r2   = r2_score(y_test, y_pred_stack)

print('=== Stacking Results ===')
print(f'RF alone  RMSE: ${results_df[results_df["Model"]=="Random Forest"]["RMSE"].values[0]:>8,.0f}')
print(f'LGB alone RMSE: ${results_df[results_df["Model"]=="LightGBM"]["RMSE"].values[0]:>8,.0f}')
print(f'Stack     RMSE: ${stack_rmse:>8,.0f}   R²={stack_r2:.4f}')
print()
print(f'Meta-learner weights: RF={meta_model.coef_[0]:.3f}, LGB={meta_model.coef_[1]:.3f}')
print('(These tell you which base model the meta-learner trusts more)')

In [ ]:
# Actual vs Predicted plot for the stacking ensemble
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_test / 1000, y_pred_stack / 1000, alpha=0.3, s=20, color='#4E79A7')
lims = [min(y_test.min(), y_pred_stack.min()) / 1000, 
        max(y_test.max(), y_pred_stack.max()) / 1000]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Price ($000s)')
ax.set_ylabel('Predicted Price ($000s)')
ax.set_title(f'Stacking Ensemble\nRMSE=${stack_rmse:,.0f}  R²={stack_r2:.4f}', fontweight='bold')
ax.legend()

ax = axes[1]
residuals = y_test - y_pred_stack
ax.hist(residuals / 1000, bins=40, color='#F28E2B', edgecolor='white', linewidth=0.5)
ax.axvline(0, color='red', linestyle='--')
ax.set_xlabel('Residual ($000s)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution\n(centred at 0 = unbiased)', fontweight='bold')

plt.tight_layout()
plt.savefig('stacking_results.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart description.** Left: predicted against actual price for the stacking ensemble, in thousands of dollars, with a dashed line for perfect prediction. RMSE is $29,535 and R² is 0.935. The points follow the line closely, spreading a little more at high prices. Right: a histogram of residuals, centred on 0 and mostly within ±50 thousand dollars.

---

## What You Built

You built a complete ensemble benchmark. You generated a realistic synthetic dataset, compared four ensemble methods (Random Forest, Gradient Boosting, XGBoost, LightGBM) on speed and accuracy, visualised feature importances, plotted the accuracy-vs-ensemble-size curve, and built a stacking ensemble that combines two base models with a Ridge meta-learner. The stack (RMSE ~$29,500) did *not* beat LightGBM on its own (~$28,300). One base model was much weaker, and the meta-learner even gave it a negative weight. Stacking only helps when the base models are strong and make different mistakes, so always compare a stack against the best single model.

## What's Next

- Read `explanation.md` if you haven't. The bagging vs. boosting distinction explains why the curves have different shapes.
- Open `howto.md` for hyperparameter tuning recipes.
- **Extension challenge:** add `CatBoostRegressor` (from the `catboost` package) to the benchmark. It handles categorical features natively, so try passing the raw string columns instead of label-encoded ones. Then try a stack of the three boosting models and see whether it beats LightGBM alone.

**Next module:** `13_cross_validation_model_selection`. Your benchmark results are only trustworthy if you evaluate them properly.

---

**AWS SageMaker version:** SageMaker has a built-in XGBoost container, the fastest path to production for gradient boosting at scale. See Module 26 for the SageMaker Training Job setup.